# Fast reads of remote NetCDF: eager reader + cached store

Reading a NetCDF over HTTPS is slow for one reason: h5py walks the HDF5 metadata with
many small, scattered, *sequentially dependent* reads. Over a ~250 ms link each one is
a separate round-trip. The default reader also discards its buffer on every seek, so a
649 MiB CMIP6 file costs ~648 separate range GETs.

Two `obspec-utils` pieces fix it:

- **`EagerStoreReader`** — fetch the object with concurrent range requests, then let
  h5py parse from memory. Trades bytes for round-trips.
- **`CachingReadableStore`** — fetch each object once and serve every later range from
  RAM. This matters more than it looks: loading the `time` coordinate costs **12,606
  range requests**, because `time` is chunked one chunk per timestep.

Measured on the file below (CEDA, 649 MiB, ~11 MB/s link):

| | default | eager + cached |
|---|---:|---:|
| metadata walk | 346 s | 59 s |
| loading coordinates | 302 s | 1.2 s |
| **end to end** | **414 s** | **58 s** |

In [ ]:
import time

import xarray as xr
from obspec_utils.readers import EagerStoreReader
from obspec_utils.wrappers import CachingReadableStore
from obstore.store import from_url

HOST = "https://dap.ceda.ac.uk"
PATH = (
    "badc/cmip6/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp585/r1i1p1f2/"
    "AERday/zg500/gn/v20190726/"
    "zg500_AERday_UKESM1-0-LL_ssp585_r1i1p1f2_gn_20150101-20491230.nc"
)
URL = f"{HOST}/{PATH}"

# One cached store, reused everywhere below. max_size must exceed the file size,
# or the object is evicted and refetched.
store = CachingReadableStore(from_url(HOST), max_size=2 * 1024**3)

## 1. `xr.open_dataset`

`EagerStoreReader` implements the `ReadableFile` protocol (`read`/`seek`/`tell`), which
is exactly what `h5netcdf` wants — so it substitutes for the URL string with no other
changes.

In [ ]:
t = time.perf_counter()
ds = xr.open_dataset(EagerStoreReader(store, PATH), engine="h5netcdf")
print(f"{time.perf_counter() - t:.1f}s")
ds

## 2. `open_virtual_dataset`

virtualizarr's `HDFParser` hardcodes its own reader, so swapping in the eager one means
a small parser of our own. A parser is just a callable `(url, registry) -> ManifestStore`.

Pass the **same cached store** in the registry — that is what makes the coordinate load
cheap.

In [ ]:
from virtualizarr.manifests import ManifestStore
from virtualizarr.parsers.hdf.hdf import _construct_manifest_group
from virtualizarr.registry import ObjectStoreRegistry
from virtualizarr.xarray import open_virtual_dataset


class EagerHDFParser:
    """HDFParser, but reading through an EagerStoreReader."""

    def __call__(self, url, registry):
        store, path = registry.resolve(url)
        reader = EagerStoreReader(store, path)
        try:
            group = _construct_manifest_group(filepath=url, reader=reader)
        finally:
            reader.close()
        return ManifestStore(group, registry=registry)


registry = ObjectStoreRegistry({HOST: store})

t = time.perf_counter()
vds = open_virtual_dataset(url=URL, registry=registry, parser=EagerHDFParser())
print(f"{time.perf_counter() - t:.1f}s")
vds

That second cell is fast because the cache already holds the file from cell 1. Build a
fresh `CachingReadableStore` to time it cold.

For multiple files use `open_virtual_mfdataset(urls=..., parser=EagerHDFParser(),
registry=registry)`. `parallel=ThreadPoolExecutor` works; `parallel=ProcessPoolExecutor`
is documented but raises `AttributeError: Can't get local object
'open_virtual_mfdataset.<locals>._open'` — it dispatches a closure that `pickle` cannot
serialize. Use `parallel="dask"` if you need processes.

Parallelism is unlikely to help over a saturated link anyway: this one held ~11 MB/s
whether driven by 1 or 128 in-flight requests, and the eager fetch already uses it all.

## The baseline, for comparison

Uncached store + stock parser. Takes ~6 minutes — run it only if you want to see the
difference yourself.

In [ ]:
from virtualizarr.parsers import HDFParser

plain = ObjectStoreRegistry({HOST: from_url(HOST)})

t = time.perf_counter()
slow = open_virtual_dataset(url=URL, registry=plain, parser=HDFParser())
print(f"{time.perf_counter() - t:.1f}s")